# 03 — Product Score Index (PSI)

Este notebook calcula o PSI e exporta rankings gerais e por categoria para `reports/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.scoring import PSIWeights, add_psi_column


## Carregar base de produtos

In [ ]:
df_products = pd.read_csv(PROCESSED_DIR / 'base_produtos.csv')
df_products.shape


## Calcular PSI

In [ ]:
weights = PSIWeights(rating=0.40, rating_count_log=0.35, discount_pct=0.25)
df_products_psi = add_psi_column(df_products, weights=weights)
df_products_psi[['product_name','main_category','rating_clean','rating_count_clean','discount_pct_clean','PSI']].head(5)


## Rankings e exportações

In [ ]:
df_products_psi = df_products_psi.sort_values('PSI', ascending=False).reset_index(drop=True)
top10 = df_products_psi.head(10)
bottom10 = df_products_psi.tail(10)
top10[['product_id','product_name','main_category','discounted_price_clean','discount_pct_clean','rating_clean','rating_count_clean','PSI']]


In [ ]:
df_products_psi.to_csv(PROCESSED_DIR / 'base_produtos_psi.csv', index=False)
top10.to_csv(REPORTS_DIR / 'psi_top10_overall.csv', index=False)
bottom10.to_csv(REPORTS_DIR / 'psi_bottom10_overall.csv', index=False)

top10_by_cat = (
    df_products_psi
    .sort_values(['main_category','PSI'], ascending=[True, False])
    .groupby('main_category', as_index=False)
    .head(10)
)
top10_by_cat.to_csv(REPORTS_DIR / 'psi_top10_by_category.csv', index=False)
(PROCESSED_DIR / 'base_produtos_psi.csv', REPORTS_DIR / 'psi_top10_by_category.csv')
